In [ ]:
# ================================================================
# TELE-SAP ANALYSIS — Q6 NOTEBOOK — COLAB VERSION
# Direct file read from Colab
# Figure texts, exported tables, and outputs translated to English
# ================================================================

# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import warnings

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA
# ================================================================
file_path = "Table1.csv"

def load_csv() -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    return df


# ================================================================
# 2. TEXT COLUMN DETECTION
# ================================================================
def find_text_columns(df: pd.DataFrame):
    return [
        c for c in df.columns
        if any(p in c.lower() for p in ["conduta", "evolucao", "plano", "obs"])
    ]


def find_main_outcome_column(df: pd.DataFrame):
    return next(
        (c for c in df.columns if any(p in c.lower() for p in ["conduta", "plano", "evolucao"])),
        None
    )


# ================================================================
# 3. FIGURE 1 — ROOT CAUSES OF NON-RESOLUTION
# ================================================================
def figure_1_root_causes_non_resolution(df: pd.DataFrame) -> None:
    keywords = {
        "Exam Requests": ["exame", "laborat", "sangue", "rx", "raio-x", "ultrassom", "solicito"],
        "In-Person Referral": ["presencial", "ubs", "upa", "exame fisico", "palpação"],
        "Specialist Referral": ["especialista", "encaminho", "parecer", "cardiologista"],
        "Scheduled Follow-up": ["retorno", "voltar", "dias", "reavaliar"],
    }

    text_cols = find_text_columns(df)

    if not text_cols:
        print("No clinical text columns were found for the root cause analysis.")
        return

    results = {}
    for category, terms in keywords.items():
        count = 0
        for col in text_cols:
            count += df[col].astype(str).str.contains("|".join(terms), case=False, na=False).sum()
        results[category] = count

    summary = pd.Series(results).sort_values(ascending=False)

    plt.figure(figsize=(12, 6))
    sns.set_theme(style="whitegrid")

    ax = sns.barplot(x=summary.values, y=summary.index, palette="flare")
    plt.title(
        "Reasons for Non-Resolution at First Contact",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )
    plt.xlabel("Number of Mentions (Frequency in Clinical Conduct)", fontsize=12)
    plt.ylabel("")

    for container in ax.containers:
        ax.bar_label(container, padding=4, fontweight="bold", color="#333333", fontsize=11)

    sns.despine(right=True, top=True)
    plt.tight_layout()
    plt.show()

    df_table = summary.reset_index()
    df_table.columns = ["Reason for Non-Resolution", "Mention Frequency"]

    total_mentions = df_table["Mention Frequency"].sum()
    if total_mentions > 0:
        df_table["Proportion (%)"] = ((df_table["Mention Frequency"] / total_mentions) * 100).round(1)
    else:
        df_table["Proportion (%)"] = 0.0

    output_file = "Table_Reasons_for_Non_Resolution_EN.csv"
    df_table.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("EXECUTIVE DIAGNOSIS: LIMITERS OF RESOLUTION CAPACITY")
    print("=" * 80)
    if not summary.empty and summary.iloc[0] > 0:
        print(f"The main barrier to immediate resolution is: {summary.index[0]} (n={summary.iloc[0]})")
    else:
        print("No mentions were found for the categories searched.")
    print("-" * 80)
    print(f"Exported table saved successfully as: '{output_file}'")
    print("=" * 80 + "\n")


# ================================================================
# 4. OUTCOME CLASSIFICATION
# ================================================================
def classify_reason(text):
    t = str(text).lower()

    if len(t) < 5 or "não compareceu" in t:
        return "Other"

    if any(k in t for k in ["presencial", "exame fisico", "upa"]):
        return "In-Person Referral"
    if any(k in t for k in ["especialista", "encaminho", "parecer"]):
        return "Specialist Referral"
    if any(k in t for k in ["exame", "laborat", "sangue", "rx"]):
        return "Exam Requests"
    if any(k in t for k in ["retorno", "reavaliar", "agendar", "voltar"]):
        return "Scheduled Follow-up"

    return "Resolved / Discharge"


def map_flow(text):
    t = str(text).lower()

    if len(t) < 5:
        return "Resolved / Discharge"

    if any(k in t for k in ["presencial", "exame fisico", "upa"]):
        return "In-Person Referral"
    if any(k in t for k in ["especialista", "encaminho", "parecer"]):
        return "Specialist Referral"
    if any(k in t for k in ["exame", "laborat", "sangue", "rx"]):
        return "Exam Requests"
    if any(k in t for k in ["retorno", "reavaliar", "agendar", "voltar"]):
        return "Scheduled Follow-up"

    return "Resolved / Discharge"


# ================================================================
# 5. FIGURE 2 — TELE-SAP RESOLUTION FLOW (SANKEY)
# ================================================================
def figure_2_resolution_flow_sankey(df: pd.DataFrame) -> None:
    df_ev = df.dropna(subset=["Record ID"]).copy()
    col_text = find_main_outcome_column(df_ev)

    if not col_text:
        print("No conduct/plan/evolution column was found for the Sankey analysis.")
        return

    df_ev["Outcome"] = df_ev[col_text].apply(classify_reason)

    n_total_base = len(df_ev)
    v_followup = df_ev[df_ev["Outcome"] == "Scheduled Follow-up"].shape[0]
    v_exams = df_ev[df_ev["Outcome"] == "Exam Requests"].shape[0]
    v_specialist = df_ev[df_ev["Outcome"] == "Specialist Referral"].shape[0]
    v_inperson = df_ev[df_ev["Outcome"] == "In-Person Referral"].shape[0]

    n_unresolved = v_followup + v_exams + v_specialist + v_inperson
    n_resolved = n_total_base - n_unresolved

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=20,
                    thickness=20,
                    line=dict(color="black", width=0.5),
                    label=[
                        f"TOTAL CONSULTATIONS<br>(N={n_total_base})",
                        f"RESOLVED / DISCHARGE<br>({n_resolved})",
                        f"NOT RESOLVED<br>({n_unresolved})",
                        f"SCHEDULED FOLLOW-UP<br>({v_followup})",
                        f"EXAM REQUESTS<br>({v_exams})",
                        f"SPECIALIST REFERRAL<br>({v_specialist})",
                        f"IN-PERSON REFERRAL<br>({v_inperson})",
                    ],
                    color=["#2c3e50", "#27ae60", "#e67e22", "#D99C88", "#C3777C", "#9A596E", "#623D5B"],
                ),
                link=dict(
                    source=[0, 0, 2, 2, 2, 2],
                    target=[1, 2, 3, 4, 5, 6],
                    value=[n_resolved, n_unresolved, v_followup, v_exams, v_specialist, v_inperson],
                    color="rgba(200, 200, 200, 0.3)",
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="<b>TeleSAP Resolution Flow (Synchronized Data)</b>",
        font_size=12
    )
    fig.show()

    sankey_table = pd.DataFrame(
        [
            {"Source": "Total Consultations", "Target": "Resolved / Discharge", "Value": n_resolved},
            {"Source": "Total Consultations", "Target": "Not Resolved", "Value": n_unresolved},
            {"Source": "Not Resolved", "Target": "Scheduled Follow-up", "Value": v_followup},
            {"Source": "Not Resolved", "Target": "Exam Requests", "Value": v_exams},
            {"Source": "Not Resolved", "Target": "Specialist Referral", "Value": v_specialist},
            {"Source": "Not Resolved", "Target": "In-Person Referral", "Value": v_inperson},
        ]
    )

    sankey_file = "TeleSAP_Resolution_Flow_Sankey_Table_EN.csv"
    sankey_table.to_csv(sankey_file, index=False, encoding="utf-8-sig")

    print("\n--- DATA VALIDATION ---")
    print(f"Total Base (N): {n_total_base}")
    print(f"Not Resolved (sum of categories): {n_unresolved}")
    print(f"Resolved / Discharge: {n_resolved}")
    print(f"Mass balance check: {n_total_base == (n_resolved + n_unresolved)}")
    print(f"Exported Sankey table: '{sankey_file}'\n")


# ================================================================
# 6. FIGURE 3 — SANKEY BASE TABLE
# ================================================================
def figure_3_sankey_base_table(df: pd.DataFrame) -> None:
    df_base = df.dropna(subset=["Record ID"]).copy()
    n_total = len(df_base)

    col_text = find_main_outcome_column(df_base)
    if not col_text:
        print("No conduct/plan/evolution column was found for Sankey base extraction.")
        return

    df_base["Outcome"] = df_base[col_text].apply(map_flow)

    summary = df_base["Outcome"].value_counts()

    v_followup = summary.get("Scheduled Follow-up", 0)
    v_exams = summary.get("Exam Requests", 0)
    v_specialist = summary.get("Specialist Referral", 0)
    v_inperson = summary.get("In-Person Referral", 0)
    v_unresolved = v_followup + v_exams + v_specialist + v_inperson
    v_resolved = n_total - v_unresolved

    base_sankey = pd.DataFrame(
        [
            {"Source": "Total Consultations", "Target": "Resolved / Discharge", "Value": v_resolved},
            {"Source": "Total Consultations", "Target": "Not Resolved", "Value": v_unresolved},
            {"Source": "Not Resolved", "Target": "Scheduled Follow-up", "Value": v_followup},
            {"Source": "Not Resolved", "Target": "Exam Requests", "Value": v_exams},
            {"Source": "Not Resolved", "Target": "Specialist Referral", "Value": v_specialist},
            {"Source": "Not Resolved", "Target": "In-Person Referral", "Value": v_inperson},
        ]
    )

    print("-" * 60)
    print(f"SANKEY BASE TABLE GENERATED (N={n_total})")
    print("-" * 60)
    print(base_sankey.to_string(index=False))
    print("-" * 60)
    print(
        f"Output sum: {v_resolved + v_unresolved} "
        f"(Matches Total: {'YES' if v_resolved + v_unresolved == n_total else 'NO'})"
    )

    output_file = "Sankey_Diagram_Base_EN.csv"
    base_sankey.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Exported base table: '{output_file}'\n")


# ================================================================
# 7. RUN EVERYTHING
# ================================================================
def main():
    df = load_csv()
    figure_1_root_causes_non_resolution(df)
    figure_2_resolution_flow_sankey(df)
    figure_3_sankey_base_table(df)


main()